# 3.1b Agents, tools and protocols

**Story:** SafeDrive's claims assistant must do more than answer from documents. It has to *look up a
real claim* and later work with other agents. In this chapter you build the same assistant three ways:
a managed **Bedrock Agent**, an open-source **Strands** agent, and that Strands agent hosted on
**Bedrock AgentCore Runtime**. Then you connect tools with **MCP** and agents with **A2A**.

**Task 3.1** Manage deployment infrastructure for ML and AI model types.
**Task 3.2** Provision and configure resources for ML and AI workloads.

| Skill | What | Status |
|---|---|---|
| 3.1.6 | Deploy and configure agents, integrate tools, agent communication protocols | ✅ |
| 3.2.11 | Deploy agentic workflow infrastructure (AgentCore Runtime, Gateway) | ✅ |

## Key points for the exam

- An **agent** = FM + instructions + tools + a loop (think → call a tool → read the result → answer).
- **Bedrock Agents**: fully managed and configuration-based. Tools are **action groups** (Lambda or
  "return control"), knowledge bases can be attached, versions and aliases are built in.
- **Bedrock AgentCore**: managed *infrastructure* for agents written in **any framework** (Strands,
  LangGraph, CrewAI ...) with **any model**. You bring the code, AWS runs it.
- **Strands Agents**: open-source Python SDK from AWS. Model-driven loop, tools are Python functions.
- **MCP** (Model Context Protocol) connects an **agent to tools**. **A2A** (Agent2Agent) connects an
  **agent to another agent**.
- **AgentCore Gateway** turns Lambda functions, OpenAPI/Smithy APIs and MCP servers into one MCP endpoint.
- A Bedrock Agent must be **prepared** after every change, and callers use an **alias**, not the draft.
- Multi-agent: a **supervisor** agent splits the work and calls **collaborator** agents.

| AgentCore part | What it gives you |
|---|---|
| Runtime | Serverless hosting, one isolated microVM per session, up to 8 hours, HTTP / MCP / A2A servers |
| Gateway | Tools behind one MCP endpoint, inbound auth (IAM or JWT), outbound credentials |
| Memory | Short-term events and long-term extracted memories (see 3.2b) |
| Identity | Workload identity for agents, OAuth and API-key credential providers |
| Observability | Traces, metrics and logs in CloudWatch (OpenTelemetry) |
| Built-in tools | Code Interpreter and Browser sandboxes |

| Choose ... | When ... |
|---|---|
| Bedrock Agents | You want the least code and a managed agent with Lambda tools and knowledge bases |
| Strands (or LangGraph, CrewAI) on AgentCore Runtime | You need custom agent logic, any framework or model, long sessions |
| AgentCore Gateway | Existing APIs or Lambda functions must become MCP tools with central auth |
| Step Functions / Bedrock Flows | The steps are fixed and known in advance (a workflow, not an agent) |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, uuid
from common import utils as U, catchup as C
os.makedirs("scripts/agentcore_agent", exist_ok=True)
print(U.REGION, U.bucket())

Catch-up: the raw data and the DynamoDB table `safedrive-claims` that the tool reads.

In [ ]:
C.ensure_raw_data()
CLAIMS_TABLE = C.ensure_claims_table()
print("claims table:", CLAIMS_TABLE)

Clients used in this notebook.

In [ ]:
lam = U.client("lambda")
agent = U.client("bedrock-agent")
rt = U.client("bedrock-agent-runtime")
ac = U.client("bedrock-agentcore-control")
acd = U.client("bedrock-agentcore")
ROLE = U.role_arn()
print(ROLE)

## 3.1.6 Part 1: Bedrock Agent with a Lambda tool

- Build order: **Lambda tool → agent → action group → prepare → alias → invoke**.
- An **action group** tells the agent what the tool does (function schema or OpenAPI schema) and who
  runs it (a Lambda function, or `RETURN_CONTROL` to hand the call back to your app).
- The Lambda needs a **resource-based permission** for `bedrock.amazonaws.com`.
- Names match `C.ensure_tool_lambda` / `C.ensure_agent`, so every step first checks "does it exist?".

### Step 1: the tool (Lambda function)
Read the tool code. Look at the response shape that Bedrock Agents expects (`messageVersion`,
`functionResponse`).

In [ ]:
src_path = os.path.join(U.REPO_ROOT, "common", "lambda_src", "claim_lookup.py")
print(open(src_path).read())

Zip the file in memory. A Lambda deployment package is just a zip.

In [ ]:
import io, zipfile
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write(src_path, "claim_lookup.py")
print("zip size (bytes):", len(buf.getvalue()))

Create the function `safedrive-claim-lookup` only if it does not exist yet.

In [ ]:
try:
    tool_lambda_arn = lam.get_function(FunctionName=C.TOOL_LAMBDA)["Configuration"]["FunctionArn"]
    print("reusing", tool_lambda_arn)
except lam.exceptions.ResourceNotFoundException:
    tool_lambda_arn = lam.create_function(
        FunctionName=C.TOOL_LAMBDA, Runtime="python3.12", Role=ROLE,
        Handler="claim_lookup.handler", Code={"ZipFile": buf.getvalue()}, Timeout=15,
        Environment={"Variables": {"CLAIMS_TABLE": CLAIMS_TABLE}},
        Tags={t["Key"]: t["Value"] for t in U.TAGS})["FunctionArn"]
    lam.get_waiter("function_active_v2").wait(FunctionName=C.TOOL_LAMBDA)
    print("created", tool_lambda_arn)

Allow Bedrock to invoke the function. "skipped ... already exists" is fine on a second run.

In [ ]:
U.safe(lam.add_permission, FunctionName=C.TOOL_LAMBDA, StatementId="bedrock-agent-invoke",
       Action="lambda:InvokeFunction", Principal="bedrock.amazonaws.com",
       SourceAccount=U.account_id())
U.save_state(tool_lambda_arn=tool_lambda_arn)
print(tool_lambda_arn)

Test the tool alone before any agent uses it. You should see one claim with its status.

In [ ]:
resp = lam.invoke(FunctionName=C.TOOL_LAMBDA, Payload=json.dumps({"claim_id": "C0000001"}))
print(json.loads(resp["Payload"].read()))

### Step 2: the agent
Reuse the agent if state or the account already has `safedrive-claims-agent`.

In [ ]:
agent_id = U.get_state("agent_id")
if not agent_id:
    found = [a for a in agent.list_agents(maxResults=100)["agentSummaries"] if a["agentName"] == C.AGENT_NAME]
    agent_id = found[0]["agentId"] if found else None
print("existing agent:", agent_id)

Create the agent: a model, a role and an **instruction** (the agent's system prompt).

In [ ]:
INSTRUCTION = ("You are the SafeDrive Insurance claims assistant. Use the claim lookup tool to "
               "answer questions about a claim. Be brief and never invent claim details.")
if not agent_id:
    agent_id = agent.create_agent(
        agentName=C.AGENT_NAME, agentResourceRoleArn=ROLE, foundationModel=U.TEXT_MODEL,
        idleSessionTTLInSeconds=900, instruction=INSTRUCTION,
        tags={t["Key"]: t["Value"] for t in U.TAGS})["agent"]["agentId"]
print("agent id:", agent_id)

Wait until the agent leaves `CREATING`. A new agent is `NOT_PREPARED`.

In [ ]:
agent_status = lambda: agent.get_agent(agentId=agent_id)["agent"]["agentStatus"]  # noqa: E731
U.wait_until(agent_status, ok=("NOT_PREPARED", "PREPARED"), bad=("FAILED",), delay=5, label="agent")

### Step 3: the action group
The **function schema** is what the model reads to decide when to call the tool and with which
parameters. Good descriptions matter more than code here.

In [ ]:
groups = agent.list_agent_action_groups(agentId=agent_id, agentVersion="DRAFT")["actionGroupSummaries"]
if not any(g["actionGroupName"] == "claims-tools" for g in groups):
    agent.create_agent_action_group(
        agentId=agent_id, agentVersion="DRAFT", actionGroupName="claims-tools",
        actionGroupExecutor={"lambda": tool_lambda_arn},
        functionSchema={"functions": [{
            "name": "get_claim_status", "description": "Look up a SafeDrive claim by its claim ID.",
            "parameters": {"claim_id": {"type": "string", "required": True,
                                        "description": "Claim ID such as C0000001"}}}]})
groups = agent.list_agent_action_groups(agentId=agent_id, agentVersion="DRAFT")["actionGroupSummaries"]
print([(g["actionGroupName"], g["actionGroupState"]) for g in groups])

### Step 4: prepare
`prepare_agent` packages the DRAFT so it can be invoked. Run it after **every** change.

In [ ]:
if agent_status() != "PREPARED":
    agent.prepare_agent(agentId=agent_id)
U.wait_until(agent_status, ok=("PREPARED",), bad=("FAILED",), delay=5, label="agent")

### Step 5: alias
Creating an alias without a routing configuration **creates a new numbered version** and points the
alias to it. Applications call the alias `live`.

In [ ]:
aliases = agent.list_agent_aliases(agentId=agent_id)["agentAliasSummaries"]
live = [a for a in aliases if a["agentAliasName"] == "live"]
if live:
    agent_alias_id = live[0]["agentAliasId"]
else:
    agent_alias_id = agent.create_agent_alias(
        agentId=agent_id, agentAliasName="live")["agentAlias"]["agentAliasId"]
print("alias id:", agent_alias_id)

In [ ]:
alias_status = lambda: agent.get_agent_alias(  # noqa: E731
    agentId=agent_id, agentAliasId=agent_alias_id)["agentAlias"]["agentAliasStatus"]
U.wait_until(alias_status, ok=("PREPARED",), bad=("FAILED",), delay=5, label="agent alias")
U.save_state(agent_id=agent_id, agent_alias_id=agent_alias_id)
print("saved agent_id and agent_alias_id")

### Step 6: invoke the agent and read the trace
`invoke_agent` returns an **event stream**: `chunk` events carry the answer, `trace` events show the
reasoning steps when `enableTrace=True`. This helper collects both.

In [ ]:
def ask_agent(text, session_id=None, alias_id=None, agent_id_=None, **kwargs):
    """Call a Bedrock agent; return (answer, list of trace parts)."""
    resp = rt.invoke_agent(agentId=agent_id_ or agent_id, agentAliasId=alias_id or agent_alias_id,
                           sessionId=session_id or str(uuid.uuid4()), inputText=text,
                           enableTrace=True, **kwargs)
    answer, traces = "", []
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode()
        if "trace" in event:
            traces.append(event["trace"]["trace"])
    return answer, traces

Ask about a claim. The answer must contain the status stored in DynamoDB.

In [ ]:
answer, traces = ask_agent("What is the status of claim C0000001 and how much was claimed?")
print(answer)
print("trace parts:", len(traces))

Read the trace: the model's rationale, the tool call (action group, function, parameters) and the
tool result. This is the first place to look when an agent picks the wrong tool.

In [ ]:
for t in traces:
    o = t.get("orchestrationTrace", {})
    if "rationale" in o:
        print("THINK  :", o["rationale"]["text"][:200])
    if "invocationInput" in o:
        call = o["invocationInput"].get("actionGroupInvocationInput", {})
        print("ACT    :", call.get("actionGroupName"), call.get("function"), call.get("parameters"))
    if "observation" in o:
        obs = o["observation"]
        out = obs.get("actionGroupInvocationOutput", {}).get("text") or obs.get("finalResponse", {}).get("text")
        print("OBSERVE:", str(out)[:200])

Token usage per model call is also in the trace (`modelInvocationOutput.metadata.usage`).

In [ ]:
usage = [t["orchestrationTrace"]["modelInvocationOutput"]["metadata"]["usage"]
         for t in traces if "modelInvocationOutput" in t.get("orchestrationTrace", {})]
print(usage)

## 3.1.6 Part 2: the same assistant as a Strands agent (runs locally)

- Strands is a Python SDK: `Agent(model, tools, system_prompt)`. The model decides which tool to call.
- A tool is a Python function with the `@tool` decorator. The **docstring and type hints** become the
  tool description and input schema.
- It calls Bedrock (Converse API) from wherever the code runs: here, your notebook.
- Needs `pip install strands-agents` (in `requirements.txt`).

Define the tool: a plain function that reads one item from DynamoDB.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

claims = U.session.resource("dynamodb").Table(CLAIMS_TABLE)

@tool
def get_claim_status(claim_id: str) -> dict:
    """Look up a SafeDrive claim by its claim ID (for example C0000001)."""
    item = claims.get_item(Key={"claim_id": claim_id}).get("Item")
    return {k: str(v) for k, v in item.items()} if item else {"found": False, "claim_id": claim_id}

print(get_claim_status.tool_spec)

Create the agent. `callback_handler=None` turns off token-by-token printing.

In [ ]:
model = BedrockModel(model_id=U.TEXT_MODEL, boto_session=U.session, temperature=0.2)
claims_agent = Agent(
    model=model, tools=[get_claim_status], callback_handler=None,
    system_prompt="You are the SafeDrive claims assistant. Use tools for claim facts. Be brief.")
print(claims_agent.tool_names)

Ask the same question. The result object prints as the final answer text.

In [ ]:
result = claims_agent("What is the status of claim C0000001 and how much was claimed?")
print(result)

Inspect the loop: how many model cycles, which tools were used, and total tokens.

In [ ]:
summary = result.metrics.get_summary()
print("cycles:", summary["total_cycles"])
print("tools :", list(summary["tool_usage"].keys()))
print("tokens:", summary["accumulated_usage"])

The conversation (including `toolUse` and `toolResult` blocks) is kept in `agent.messages`.

In [ ]:
for m in claims_agent.messages:
    kinds = [next(iter(block)) for block in m["content"]]
    print(m["role"], kinds)

## 3.2.11 Deploy the Strands agent to AgentCore Runtime

- Runtime hosts your agent code. The contract is simple: an HTTP server on port **8080** with
  `POST /invocations` and `GET /ping`. `BedrockAgentCoreApp` from the `bedrock-agentcore` SDK gives you that.
- Two ways to ship code:

| Way | How | Needs |
|---|---|---|
| **Direct code deployment** (used here) | zip of code + **arm64** Linux dependencies in S3 | pip only, zip ≤ 250 MB |
| **Container** | arm64 image in ECR | Docker locally, or CodeBuild via the starter toolkit (`agentcore configure` + `agentcore launch`) |

- Each `runtimeSessionId` gets its own isolated microVM. No idle cost: you pay for CPU and memory used.
- Every update creates a new immutable **version**. Callers reach a version through an **endpoint**
  (`DEFAULT` always points to the latest version).

### Step 1: agent code
The same Strands agent, wrapped in `BedrockAgentCoreApp`. `@app.entrypoint` is the function Runtime
calls for each request.

In [ ]:
%%writefile scripts/agentcore_agent/main.py
"""SafeDrive claims agent (Strands) for Amazon Bedrock AgentCore Runtime."""
import os

import boto3
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

REGION = os.environ.get("AWS_REGION", "us-east-1")
app = BedrockAgentCoreApp()
table = boto3.resource("dynamodb", region_name=REGION).Table(os.environ.get("CLAIMS_TABLE", "safedrive-claims"))


@tool
def get_claim_status(claim_id: str) -> dict:
    """Look up a SafeDrive claim by its claim ID (for example C0000001)."""
    item = table.get_item(Key={"claim_id": claim_id}).get("Item")
    return {k: str(v) for k, v in item.items()} if item else {"found": False, "claim_id": claim_id}


agent = Agent(
    model=BedrockModel(model_id=os.environ.get("MODEL_ID", "us.amazon.nova-lite-v1:0"), region_name=REGION),
    tools=[get_claim_status], callback_handler=None,
    system_prompt="You are the SafeDrive claims assistant. Use tools for claim facts. Be brief.")


@app.entrypoint
def invoke(payload):
    """Runtime calls this with the JSON body of the request."""
    result = agent(payload.get("prompt", "Hello"))
    return {"result": str(result)}


if __name__ == "__main__":
    app.run()

Dependencies of the agent. They are installed into the zip, not into your notebook kernel.

In [ ]:
%%writefile scripts/agentcore_agent/requirements.txt
strands-agents
bedrock-agentcore
boto3

Optional local test (in a terminal): `python scripts/agentcore_agent/main.py`, then
`curl -X POST localhost:8080/invocations -H "Content-Type: application/json" -d '{"prompt":"Status of C0000001?"}'`.

### Step 2: build the deployment package
Runtime runs on **arm64**. pip downloads Linux arm64 wheels into `build/agentcore_pkg` even if your
machine is x86 or a Mac. Takes 1-2 minutes. A pip "dependency conflicts" warning about your own
environment can be ignored.

In [ ]:
import shutil
shutil.rmtree("build/agentcore_pkg", ignore_errors=True)
!{sys.executable} -m pip install -q --platform manylinux2014_aarch64 --only-binary=:all: --python-version 3.12 --implementation cp --target build/agentcore_pkg -r scripts/agentcore_agent/requirements.txt
print(len(os.listdir("build/agentcore_pkg")), "top-level entries installed")

Add `main.py` at the root of the package and zip it.

In [ ]:
shutil.copy("scripts/agentcore_agent/main.py", "build/agentcore_pkg/main.py")
zip_path = shutil.make_archive("build/agentcore_deployment", "zip", "build/agentcore_pkg")
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB (limit 250 MB)")

Upload the zip to S3. Runtime reads it from there with the execution role.

In [ ]:
RUNTIME_NAME = "safedrive_claims_strands"      # letters, digits and underscore only
code_key = f"{U.PREFIX}/agentcore/{RUNTIME_NAME}/deployment.zip"
U.client("s3").upload_file(zip_path, U.bucket(), code_key)
print(U.s3_uri(code_key))

### Step 3: create the agent runtime
Reuse it if it exists. `entryPoint` is the file to start, `runtime` the managed Python version.

In [ ]:
ARTIFACT = {"codeConfiguration": {"code": {"s3": {"bucket": U.bucket(), "prefix": code_key}},
                                  "runtime": "PYTHON_3_12", "entryPoint": ["main.py"]}}
ENV = {"MODEL_ID": U.TEXT_MODEL, "CLAIMS_TABLE": CLAIMS_TABLE, "AWS_REGION": U.REGION}
found = [r for r in ac.list_agent_runtimes(maxResults=100)["agentRuntimes"]
         if r["agentRuntimeName"] == RUNTIME_NAME]
print("existing runtime:", found[0]["agentRuntimeId"] if found else None)

In [ ]:
if found:
    runtime_id, runtime_arn = found[0]["agentRuntimeId"], found[0]["agentRuntimeArn"]
else:
    r = ac.create_agent_runtime(
        agentRuntimeName=RUNTIME_NAME, agentRuntimeArtifact=ARTIFACT, roleArn=ROLE,
        networkConfiguration={"networkMode": "PUBLIC"},
        protocolConfiguration={"serverProtocol": "HTTP"},
        lifecycleConfiguration={"idleRuntimeSessionTimeout": 300, "maxLifetime": 1800},
        environmentVariables=ENV, description="SafeDrive Strands claims agent",
        tags={t["Key"]: t["Value"] for t in U.TAGS})
    runtime_id, runtime_arn = r["agentRuntimeId"], r["agentRuntimeArn"]
print(runtime_arn)

Wait for `READY` (about a minute). If it fails, the next cell prints the reason.

In [ ]:
runtime_status = lambda: ac.get_agent_runtime(agentRuntimeId=runtime_id)["status"]  # noqa: E731
U.wait_until(runtime_status, ok=("READY",), bad=("CREATE_FAILED", "UPDATE_FAILED"), delay=10, label="runtime")

In [ ]:
info = ac.get_agent_runtime(agentRuntimeId=runtime_id)
print("version:", info["agentRuntimeVersion"], "| protocol:", info["protocolConfiguration"]["serverProtocol"])
print("failure reason:", info.get("failureReason"))
U.save_state(agentcore_runtime_arn=runtime_arn)

### Step 4: invoke the hosted agent
Use the **data plane** client `bedrock-agentcore`. The session ID must be at least 33 characters.
The first call of a session is slower (cold start of the microVM).

In [ ]:
session_id = f"safedrive-session-{uuid.uuid4()}"
resp = acd.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn, runtimeSessionId=session_id, qualifier="DEFAULT",
    contentType="application/json", accept="application/json",
    payload=json.dumps({"prompt": "What is the status of claim C0000002?"}).encode())
print(resp["contentType"], resp.get("statusCode"))
print(json.loads(resp["response"].read()))

Same session ID again: the request lands in the same microVM, so the agent still has the conversation.

In [ ]:
resp = acd.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn, runtimeSessionId=session_id, qualifier="DEFAULT",
    contentType="application/json", accept="application/json",
    payload=json.dumps({"prompt": "Which claim did I just ask about?"}).encode())
print(json.loads(resp["response"].read()))

List the endpoints. `DEFAULT` was created for you and follows the latest version.

In [ ]:
eps = ac.list_agent_runtime_endpoints(agentRuntimeId=runtime_id)["runtimeEndpoints"]
print([(e["name"], e.get("liveVersion"), e["status"]) for e in eps])

📝 **Container path (not run here).** With the starter toolkit the same agent ships as a container.
CodeBuild builds the arm64 image, so you do not need Docker locally (a local build does).

```bash
pip install bedrock-agentcore-starter-toolkit
agentcore configure --entrypoint main.py --name safedrive_claims_container --execution-role <role-arn>
agentcore launch          # CodeBuild -> ECR -> create_agent_runtime(containerConfiguration=...)
agentcore invoke '{"prompt": "Status of C0000001?"}'
```

## 3.1.6 Part 3: MCP, connecting agents to tools

- MCP is an open protocol (JSON-RPC 2.0). An MCP **server** offers tools, an MCP **client** (inside
  the agent) calls `tools/list` to discover them and `tools/call` to run one.
- Transports: **stdio** (server is a local child process) and **streamable HTTP** (remote server).
- On AWS you can host an MCP server on AgentCore Runtime (`serverProtocol="MCP"`) or let
  **AgentCore Gateway** present Lambda functions and APIs as MCP tools without writing a server.

### MCP demo A: a tiny local MCP server over stdio (always runnable)
One tool that applies the SafeDrive deductible rule (USD 250, plus USD 200 for drivers under 25).

In [ ]:
%%writefile scripts/mcp_policy_server.py
"""Tiny local MCP server (stdio transport) with one SafeDrive tool."""
try:
    from mcp.server.mcpserver import MCPServer as FastMCP   # mcp 2.x name
except ImportError:
    from mcp.server.fastmcp import FastMCP                   # mcp 1.x name

mcp = FastMCP("safedrive-policy-tools")


@mcp.tool()
def estimate_payout(claim_amount: float, driver_age: int) -> dict:
    """Estimate the payout of a claim after the SafeDrive deductible."""
    deductible = 250 + (200 if driver_age < 25 else 0)
    return {"deductible": deductible, "payout": max(claim_amount - deductible, 0)}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Build an MCP client that starts the server as a child process and talks to it over stdin/stdout.

In [ ]:
from mcp import StdioServerParameters, stdio_client
from strands.tools.mcp import MCPClient

server = StdioServerParameters(command=sys.executable, args=["scripts/mcp_policy_server.py"])
mcp_client = MCPClient(lambda: stdio_client(server, errlog=open(os.devnull, "w")))
print("client ready (server starts when the client is opened)")

Discover the tools (`tools/list`). The schema comes from the function's type hints.

In [ ]:
with mcp_client:
    mcp_tools = mcp_client.list_tools_sync()
    for t in mcp_tools:
        print(t.tool_name, "->", json.dumps(t.tool_spec["inputSchema"]["json"]["properties"]))

Call the tool directly (`tools/call`), without any model.

In [ ]:
with mcp_client:
    out = mcp_client.call_tool_sync(tool_use_id="demo-1", name="estimate_payout",
                                    arguments={"claim_amount": 1267.3, "driver_age": 22})
print(out["status"], out["content"][0]["text"])

Give the Strands agent both tools: the Python `@tool` and the MCP tool. The agent must stay inside
the `with` block because the MCP connection is open only there.

In [ ]:
with mcp_client:
    combo = Agent(model=model, tools=[get_claim_status] + mcp_client.list_tools_sync(), callback_handler=None,
                  system_prompt="You are the SafeDrive claims assistant. Use tools. Be brief.")
    result = combo("Look up claim C0000001. The driver is 22 years old. Estimate the payout.")
print(result)
print("tools used:", list(result.metrics.get_summary()["tool_usage"].keys()))

### MCP demo B: the claim-lookup Lambda as an MCP tool through AgentCore Gateway

- **Gateway** = one managed MCP endpoint. **Targets** = what is behind it (Lambda, OpenAPI, Smithy,
  another MCP server).
- **Inbound auth** (who may call the gateway): `AWS_IAM` (SigV4, used here) or `CUSTOM_JWT`
  (OAuth tokens from Cognito, Okta ...).
- **Outbound auth** (how the gateway calls the target): the gateway's IAM role for Lambda, or OAuth /
  API key credential providers for external APIs.
- Tool names become `<target name>___<tool name>`.

Create the gateway with IAM inbound auth (reuse if it exists).

In [ ]:
GATEWAY_NAME = "safedrive-claims-gateway"
gws = [g for g in ac.list_gateways(maxResults=100)["items"] if g["name"] == GATEWAY_NAME]
if gws:
    gateway_id = gws[0]["gatewayId"]
else:
    gateway_id = ac.create_gateway(
        name=GATEWAY_NAME, roleArn=ROLE, protocolType="MCP", authorizerType="AWS_IAM",
        description="SafeDrive claim tools as MCP",
        tags={t["Key"]: t["Value"] for t in U.TAGS})["gatewayId"]
print("gateway id:", gateway_id)

In [ ]:
U.wait_until(lambda: ac.get_gateway(gatewayIdentifier=gateway_id)["status"],
             ok=("READY",), bad=("FAILED",), delay=5, label="gateway")
gateway_url = ac.get_gateway(gatewayIdentifier=gateway_id)["gatewayUrl"]
print(gateway_url)

Add the Lambda as a target. The **tool schema** (JSON Schema) is what MCP clients will see.
The gateway sends the tool arguments as the Lambda `event`, which our function already accepts.

In [ ]:
TOOL_SCHEMA = [{
    "name": "get_claim_status", "description": "Look up a SafeDrive claim by its claim ID.",
    "inputSchema": {"type": "object", "required": ["claim_id"],
                    "properties": {"claim_id": {"type": "string", "description": "Claim ID such as C0000001"}}}}]
targets = ac.list_gateway_targets(gatewayIdentifier=gateway_id)["items"]
if not any(t["name"] == "claims" for t in targets):
    ac.create_gateway_target(
        gatewayIdentifier=gateway_id, name="claims",
        targetConfiguration={"mcp": {"lambda": {"lambdaArn": tool_lambda_arn,
                                                "toolSchema": {"inlinePayload": TOOL_SCHEMA}}}},
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}])
targets = ac.list_gateway_targets(gatewayIdentifier=gateway_id)["items"]
print([(t["name"], t["status"]) for t in targets])

In [ ]:
target_id = targets[0]["targetId"]
U.wait_until(lambda: ac.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)["status"],
             ok=("READY",), bad=("FAILED",), delay=5, label="gateway target")

A minimal MCP client: JSON-RPC over HTTPS, signed with **SigV4** (service name `bedrock-agentcore`).
Writing it by hand shows what MCP looks like on the wire.

In [ ]:
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

def mcp_request(method, params=None, version=None):
    """Send one MCP JSON-RPC request to the gateway and return the parsed reply."""
    body = json.dumps({"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": method, "params": params or {}})
    headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
    if version:
        headers["MCP-Protocol-Version"] = version
    signed = AWSRequest(method="POST", url=gateway_url, data=body, headers=headers)
    SigV4Auth(U.session.get_credentials(), "bedrock-agentcore", U.REGION).add_auth(signed)
    r = requests.post(gateway_url, data=body, headers=dict(signed.headers), timeout=60)
    text = r.text.strip()
    if text.startswith(("event:", "data:")):      # reply came as a server-sent event
        text = [line[5:] for line in text.splitlines() if line.startswith("data:")][-1]
    return json.loads(text)

`initialize`: the MCP handshake. The server answers with the protocol version it will use.

In [ ]:
init = mcp_request("initialize", {"protocolVersion": "2025-03-26", "capabilities": {},
                                  "clientInfo": {"name": "safedrive-notebook", "version": "1.0"}})
print(json.dumps(init, indent=2)[:600])
mcp_version = init.get("result", {}).get("protocolVersion")

`tools/list`: discover the tools. Note the `claims___` prefix.

In [ ]:
listed = mcp_request("tools/list", version=mcp_version)
for t in listed["result"]["tools"]:
    print(t["name"], "-", t["description"])

`tools/call`: the gateway invokes the Lambda with its own role and returns the result.

In [ ]:
called = mcp_request("tools/call", {"name": "claims___get_claim_status",
                                    "arguments": {"claim_id": "C0000003"}}, version=mcp_version)
print(json.dumps(called.get("result", called), indent=2)[:600])

🔒 **JWT inbound auth (notes only, needs an identity provider).** For users or partner agents outside
AWS, create the gateway with `authorizerType="CUSTOM_JWT"` and send `Authorization: Bearer <token>`:

```python
ac.create_gateway(name=..., roleArn=ROLE, protocolType="MCP", authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": "https://cognito-idp.<region>.amazonaws.com/<pool-id>/.well-known/openid-configuration",
        "allowedClients": ["<app-client-id>"]}})
```
A Strands agent then connects with `MCPClient(lambda: streamablehttp_client(gateway_url, headers={"Authorization": f"Bearer {token}"}))`.

## 3.1.6 Part 4: A2A, connecting agents to agents

- **A2A (Agent2Agent)** is an open protocol (JSON-RPC over HTTP) for agents built by different teams
  or frameworks to call each other.
- Each agent publishes an **agent card** at `/.well-known/agent-card.json`: name, description,
  skills, URL. Other agents read the card to decide whether and how to call it.
- AgentCore Runtime can host an A2A server (`serverProtocol="A2A"`, port 9000).

| | MCP | A2A |
|---|---|---|
| Connects | agent ↔ tools and data | agent ↔ agent |
| Discovery | `tools/list` | agent card |
| The other side is | a function with a schema | an autonomous agent with its own model and tools |

The demo below needs the extra `pip install "strands-agents[a2a]"`. It is local: one agent is served
over A2A in a child process, a second agent calls it.

In [ ]:
%pip install -q "strands-agents[a2a]"

In [ ]:
%%writefile scripts/a2a_policy_server.py
"""Serve a Strands agent over the A2A protocol on http://127.0.0.1:9000."""
import os

from strands import Agent
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer

agent = Agent(
    name="SafeDrive policy expert",
    description="Answers questions about SafeDrive motor policy rules such as deductibles.",
    model=BedrockModel(model_id=os.environ["MODEL_ID"], region_name=os.environ["AWS_REGION"]),
    system_prompt=("You are a SafeDrive policy expert. The standard deductible is USD 250 per claim. "
                   "Drivers under 25 pay an extra USD 200. Answer in one sentence."),
    callback_handler=None)

if __name__ == "__main__":
    A2AServer(agent=agent, host="127.0.0.1", port=9000).serve()

Start the A2A server in the background and wait a few seconds for it to come up.

In [ ]:
import subprocess
env = dict(os.environ, MODEL_ID=U.TEXT_MODEL, AWS_REGION=U.REGION)
a2a_proc = subprocess.Popen([sys.executable, "scripts/a2a_policy_server.py"], env=env,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
print("server running:", a2a_proc.poll() is None)

Fetch the **agent card**. This is all another agent needs to know to call this one.

In [ ]:
card = requests.get("http://127.0.0.1:9000/.well-known/agent-card.json", timeout=10).json()
print(json.dumps({k: card.get(k) for k in ("name", "description", "url", "protocolVersion", "skills")}, indent=2))

Call the remote agent through A2A. `A2AAgent` is the Strands client for any A2A server.

In [ ]:
from strands.agent.a2a_agent import A2AAgent
remote_expert = A2AAgent(endpoint="http://127.0.0.1:9000")
print(remote_expert("What deductible does a 22 year old driver pay?"))

Stop the A2A server.

In [ ]:
a2a_proc.terminate()
print("stopped, exit code:", a2a_proc.wait(timeout=10))

## 3.1.6 Part 5: multi-agent collaboration

- Pattern: a **supervisor** receives the request, delegates sub-tasks to specialist agents and merges
  the answers.
- **Strands**: wrap each specialist as a tool ("agents as tools"). Also has Swarm, Graph and A2A.
- **Bedrock Agents**: set `agentCollaboration` on the supervisor and attach collaborators by **alias ARN**.
  `SUPERVISOR` plans and merges, `SUPERVISOR_ROUTER` only routes simple requests (faster, cheaper).

### Strands: agents as tools
Two specialists, each wrapped in a `@tool` function. The docstring tells the supervisor when to use it.

In [ ]:
policy_expert = Agent(model=model, callback_handler=None, system_prompt=(
    "You are a SafeDrive policy expert. Standard deductible is USD 250 per claim. Be brief."))

@tool
def ask_claims_specialist(question: str) -> str:
    """Ask the claims specialist about the status or details of one claim ID."""
    return str(claims_agent(question))

@tool
def ask_policy_expert(question: str) -> str:
    """Ask the policy expert about policy rules such as deductibles and exclusions."""
    return str(policy_expert(question))

print(ask_claims_specialist.tool_name, ask_policy_expert.tool_name)

The supervisor only has the two specialists as tools. Check that both were used.

In [ ]:
supervisor = Agent(model=model, tools=[ask_claims_specialist, ask_policy_expert], callback_handler=None,
                   system_prompt="You are a supervisor. Delegate to the specialists, then give one short answer.")
result = supervisor("What is the amount of claim C0000001, and what is the standard deductible?")
print(result)
print("delegated to:", list(result.metrics.get_summary()["tool_usage"].keys()))

### Bedrock Agents: supervisor + collaborator
Create a supervisor agent (reuse if it exists). It has no tools of its own.

In [ ]:
SUPERVISOR_NAME = "safedrive-supervisor-agent"
found = [a for a in agent.list_agents(maxResults=100)["agentSummaries"] if a["agentName"] == SUPERVISOR_NAME]
if found:
    supervisor_id = found[0]["agentId"]
else:
    supervisor_id = agent.create_agent(
        agentName=SUPERVISOR_NAME, agentResourceRoleArn=ROLE, foundationModel=U.TEXT_MODEL,
        agentCollaboration="SUPERVISOR", idleSessionTTLInSeconds=600,
        instruction=("You are the SafeDrive front-desk supervisor. Send every question about a specific "
                     "claim to the claims collaborator and return its answer."),
        tags={t["Key"]: t["Value"] for t in U.TAGS})["agent"]["agentId"]
sup_status = lambda: agent.get_agent(agentId=supervisor_id)["agent"]["agentStatus"]  # noqa: E731
U.wait_until(sup_status, ok=("NOT_PREPARED", "PREPARED"), bad=("FAILED",), delay=5, label="supervisor")

Attach the claims agent as a collaborator. You must pass its **alias ARN** (not the draft).

In [ ]:
alias_arn = agent.get_agent_alias(agentId=agent_id, agentAliasId=agent_alias_id)["agentAlias"]["agentAliasArn"]
existing = agent.list_agent_collaborators(agentId=supervisor_id, agentVersion="DRAFT")["agentCollaboratorSummaries"]
if not existing:
    agent.associate_agent_collaborator(
        agentId=supervisor_id, agentVersion="DRAFT", agentDescriptor={"aliasArn": alias_arn},
        collaboratorName="claims-specialist", relayConversationHistory="TO_COLLABORATOR",
        collaborationInstruction="Use this agent to look up the status and amount of a claim by claim ID.")
print(alias_arn)

Prepare the supervisor.

In [ ]:
agent.prepare_agent(agentId=supervisor_id)
U.wait_until(sup_status, ok=("PREPARED",), bad=("FAILED",), delay=5, label="supervisor")

Invoke the supervisor's draft with the built-in test alias `TSTALIASID`. In the trace, look for the
hand-off to the collaborator.

In [ ]:
answer, traces = ask_agent("What is the status of claim C0000004?", agent_id_=supervisor_id, alias_id="TSTALIASID")
print(answer)
for t in traces:
    call = t.get("orchestrationTrace", {}).get("invocationInput", {}).get("agentCollaboratorInvocationInput")
    if call:
        print("delegated to:", call.get("agentCollaboratorName"), "|", call.get("input", {}).get("text"))

## Exam traps

- **MCP is agent-to-tool, A2A is agent-to-agent.** A question about agents from different vendors
  working together points to A2A; "expose an existing API as a tool" points to MCP / AgentCore Gateway.
- Changed a Bedrock agent and nothing happened? You forgot **`prepare_agent`**, or the alias still points
  to the old version.
- Lambda tool fails with access denied: the function needs a **resource-based policy** for
  `bedrock.amazonaws.com` (the agent role alone is not enough).
- AgentCore Runtime is **not** tied to Bedrock models or to Strands. Any framework, any model, arm64.
- Fixed, predictable steps do not need an agent. Use Step Functions or Bedrock Flows (cheaper, testable).

## Quiz

**1.** A team built a LangGraph agent that calls a model outside Bedrock. Sessions can last hours and
must be isolated per user. Which service hosts it with the least operational work?

A. Bedrock Agents  B. AgentCore Runtime  C. A SageMaker real-time endpoint  D. Lambda

<details><summary>Answer</summary>B. Runtime hosts any framework and model, gives one microVM per session and runs up to 8 hours. Lambda stops at 15 minutes.</details>

**2.** SafeDrive has 30 internal REST APIs with OpenAPI specs. Several agents must use them as tools
with central authentication. What fits best?

A. One Lambda action group per API  B. AgentCore Gateway with OpenAPI targets  C. A2A servers for each API  D. Hard-code the HTTP calls in each prompt

<details><summary>Answer</summary>B. Gateway turns APIs into MCP tools behind one endpoint and handles inbound and outbound auth.</details>

**3.** A claims agent from SafeDrive must ask a repair-shop partner's agent (other framework, other
company) for a repair quote. Which protocol is designed for this?

A. MCP  B. A2A  C. SigV4  D. gRPC streaming to a SageMaker endpoint

<details><summary>Answer</summary>B. A2A is for agent-to-agent calls, with discovery through agent cards.</details>

**4.** You added an action group to a Bedrock agent. Calls through the `live` alias still do not use
the new tool. What is the most likely reason?

A. The Lambda timeout is too low  B. The DRAFT was not prepared and the alias still routes to the old version  C. Traces are disabled  D. The model does not support tools

<details><summary>Answer</summary>B. Changes live in DRAFT. Prepare, create a new version and point the alias to it.</details>

**5.** A request often needs both a claims lookup and a policy answer, each handled by a specialist
Bedrock agent. What is the managed way to combine them?

A. A supervisor agent with collaborators (multi-agent collaboration)  B. One huge prompt with both instructions  C. Two aliases on one agent  D. An inference pipeline endpoint

<details><summary>Answer</summary>A. The supervisor delegates to collaborator agents (attached by alias ARN) and merges the results.</details>

## Save state

In [ ]:
state = U.save_state(agent_id=agent_id, agent_alias_id=agent_alias_id, tool_lambda_arn=tool_lambda_arn,
                     agentcore_runtime_arn=runtime_arn)
print({k: state[k] for k in ("agent_id", "agent_alias_id", "tool_lambda_arn", "agentcore_runtime_arn")})

## Cleanup

**Kept for later notebooks** (no idle cost): Lambda `safedrive-claim-lookup`, Bedrock agent
`safedrive-claims-agent` with alias `live`, AgentCore runtime `safedrive_claims_strands`, DynamoDB table
`safedrive-claims`. The teardown notebook removes them.

Deleted here: the gateway (target first) and the supervisor agent.

In [ ]:
U.safe(ac.delete_gateway_target, gatewayIdentifier=gateway_id, targetId=target_id)

The gateway can only be deleted after its targets are gone, so wait a little.

In [ ]:
time.sleep(20)
U.safe(ac.delete_gateway, gatewayIdentifier=gateway_id)

In [ ]:
U.safe(agent.delete_agent, agentId=supervisor_id, skipResourceInUseCheck=True)

Remove the local build folder (about 100 MB).

In [ ]:
shutil.rmtree("build", ignore_errors=True)
print("local build folder removed")